---
# Chapter 5 — Pathways Through Memory

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 5: Pathways Through Memory |
| Central question | Can recall travel through relationships? Does associative propagation buy anything that lookup cannot? |
| Main concepts | Associative propagation, Spreading activation, Lateral inhibition, Pathway traces |
| Implementation | associative_memory |
| Experiment | Associative propagation over GraphRAG snapshot |
| Evidence status | Developmental: conditional |
| Depends on | Chapter 4 (GraphRAG), Chapter 2 (instrument) |

---

## What this notebook demonstrates

This chapter tests whether **associative propagation** over the derived graph recovers multi-hop evidence that lookup misses. The notebook:

1. **Loads the GraphRAG snapshot** (from Chapter 4's frozen index)
2. **Seeds activation** from a query entity
3. **Runs spreading activation** with configurable parameters
4. **Inspects activation/path traces** and compares against seed-only control
5. **Shows the ablation results**: unconstrained propagation propagated error; lateral inhibition did not earn itself

> **Evidence status**: Developmental. Cue-conditioned propagation recovered some multi-hop evidence; unconstrained propagation and naive strengthening propagated error; lateral inhibition did not earn itself. The mechanism is conditional.

## The chapter question

> **Can recall travel through relationships? Does moving buy anything that lookup cannot?**

The graph from Chapter 4 is static — relationships sit until a query arrives. But remembering does something else: a cue touches one memory, activation moves along relations, and the needed memory arrives three steps later by a route no similarity computation planned.

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(5)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 5 Concepts")

## Load the associative memory implementation

In [ ]:
from associative_memory import AssociativeConfig, AssociativeMemory
from associative_memory.fixtures import build_graph, build_cases

graph = build_graph()
print(f"Fixture graph: {len(graph.nodes)} nodes, {len(graph.edges)} edges")
print(f"Graph version: {graph.version} (corpus: {graph.corpus_version})")
print("Retrieval strategies: direct, pagerank, spreading, conditioned, conditioned-pagerank")
print(f"Default strategy: {AssociativeConfig().propagation.strategy}")
cases = build_cases()
print(f"Probe cues: {len(cases)} cases")

## The GraphRAG snapshot adapter

Chapter 5 runs propagation over the **derived graph snapshot** from Chapter 4 through a snapshot adapter. This connects the persistent graph to associative pathways.

In [ ]:
from associative_memory import AssociativeConfig

cfg = AssociativeConfig()
print("Seeding:", cfg.seeding.method, "| top_k:", cfg.seeding.top_k,
      "| min_score:", cfg.seeding.min_score)
print("Propagation:", cfg.propagation.strategy,
      "| max_hops:", cfg.propagation.max_hops,
      "| retention:", cfg.propagation.retention,
      "| inhibition:", cfg.propagation.inhibition,
      "| frontier_top_m:", cfg.propagation.frontier_top_m)
print("Selection: max_memories:", cfg.selection.max_memories,
      "| activation/similarity/importance weights:",
      (cfg.selection.activation_weight, cfg.selection.similarity_weight,
       cfg.selection.importance_weight))

## Seed a fixture graph and run propagation

The chapter uses a real fixture graph. Let's demonstrate with the event-store example.

In [ ]:
from associative_memory import AssociativeConfig, AssociativeMemory
from associative_memory.fixtures import build_graph

graph = build_graph()
memory = AssociativeMemory(graph=graph, config=AssociativeConfig())

# Multi-hop probe: the evidence names neither the decision nor its outcome
CUE = "Why was the event-store backend changed?"
result = memory.retrieve(CUE)

print(f"cue: {result.cue}")
print(f"strategy: {result.strategy}")
print(f"seeds: {[(s.node_id, round(s.score, 3)) for s in result.seeds]}")
print(f"\nadmitted ({len(result.admitted)} of {len(result.explored)} explored):")
for m in result.admitted:
    print(f"  {m.node_id} activation={m.activation:.3f} hops={m.hops}")
    print(f"    route: {' -> '.join(m.path)}")
print(f"\nexplored={len(result.explored)} expanded={result.nodes_expanded} "
      f"edges={result.edges_traversed} stopped={result.terminated_because}")
print(f"admitted sources: {result.admitted_sources()}")
print(f"\n{memory.explain(result)[:600]}")

## Compare: Seed-only vs Propagation

The key comparison: does propagation recover evidence that the seed alone misses?

In [ ]:
# Seed-only baseline (direct neighbourhood) vs spreading, with and
# without lateral inhibition. Same cue, same graph, same budget.
from associative_memory import (
    AssociativeConfig, AssociativeMemory, PropagationConfig,
)

strategies = [
    ("direct (seed neighbourhood)", AssociativeConfig(), "direct"),
    ("spreading", AssociativeConfig(), "spreading"),
    ("spreading, no inhibition",
     AssociativeConfig(propagation=PropagationConfig(inhibition=False)),
     "spreading"),
]
for name, cfg, strategy in strategies:
    mem = AssociativeMemory(graph=graph, config=cfg, strategy=strategy)
    r = mem.retrieve(CUE)
    print(f"{name:28s} admitted={len(r.admitted):2d} explored={len(r.explored):3d} "
          f"sources={r.admitted_sources()}")

## The ablation results (from the chapter)

- **Cue-conditioned propagation**: Recovered some multi-hop evidence (e.g., contention → decision path)
- **Unconstrained propagation**: Propagated error (typo-split entities from Chapter 4 activated incorrectly)
- **Naive strengthening**: Amplified both signal and noise
- **Lateral inhibition**: Did not earn itself — no measurable contribution over simpler propagation

> **Developmental verdict**: Conditional. The mechanism is not universally beneficial; it requires careful gating to avoid amplifying the persistent mistakes from Chapter 4.

## What this establishes

- Associative propagation can compose across artifacts (multi-hop)
- But it also propagates **persistent mistakes** (typo nodes, entity splits)
- Lateral inhibition did not earn its complexity
- The mechanism is conditional — requires gating by provenance/trust
- Downstream consumers (Chapter 6 routing, Chapter 7 lineage) inherit both signal and noise

## What this does NOT establish

- No general quality win for propagation over lookup
- No frozen comparative run establishing book result
- Real-corpus extraction quality for propagation untested

## Try it yourself

Modify the seed entity or propagation steps. Try seeding from SQLITE (the superseded entity) and observe how activation flows through the SUPERSEDES edge.

In [ ]:
# TRY IT YOURSELF: a weakly-connected probe. The correct memories have
# low degree, so lookup struggles and propagation must earn its keep.
CUE2 = ("Which test fixtures were left encoding assumptions "
        "from the old backend?")
r2 = memory.retrieve(CUE2)
print(f"cue: {r2.cue}")
print(f"seeds: {[(s.node_id, round(s.score, 3)) for s in r2.seeds]}")
for m in r2.admitted:
    print(f"  {m.node_id} activation={m.activation:.3f} hops={m.hops}")
    print(f"    route: {' -> '.join(m.path)}")

## Where this leads next

Chapter 6 asks: **Who chooses how we remember?** The Memory Nexus routes the same query to multiple memory capabilities and exercises the real routing/control policy.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory-chapter.ipynb)